# La régression sans les anomalies

La détection d'anomalies du projet tourne d'abord. Les salariés qu'elle classe dans les priorités choisies (`Critical` pour commencer) sont retirés, puis les six modèles de `Prediction_Compa_Ratio` sont refaits sur ceux qui restent.

Tout ce que le notebook enregistre s'écrit dans son dossier, à côté de lui.

---
## 1. Lancer la détection d'anomalies

Les fonctions sont celles du projet (dossier `src/`), avec les réglages de `config/rules.yaml`. Chaque salarié reçoit un score et une priorité : `Critical`, `Major`, `Minor` ou `Info`.

In [ ]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import yaml
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Les fonctions de la détection sont celles du projet, dans src/ : elles ne sont pas recopiées ici
sys.path.append('../src')
from anomaly_pretraitement import preparer_employes
from anomaly_cohortes import former_cohortes
from anomaly_regles import apply_rulebook
from anomaly_signal_iforest import ml_anomaly
from anomaly_signal_fort import apply_ml_strong_signal
from anomaly_score_general import aggregate_risk
from anomaly_recommandations import recommendations

DOSSIER = '../input'   # le dossier des trois fichiers Excel
px.defaults.template = 'plotly_white'   # des graphes sur fond blanc
pd.set_option('display.precision', 2)   # les tableaux affichent 2 chiffres après la virgule ; les calculs gardent toute leur précision

# Les réglages de la détection (seuils, poids, priorités) : ceux du projet
with open('../config/rules.yaml', encoding='utf-8') as fichier:
    regles = yaml.safe_load(fichier)['rules']

# Les étapes de la détection, dans le même ordre que le programme principal (src/detect_salary_anomalies.py)
detection = preparer_employes(f'{DOSSIER}/employes.xlsx', f'{DOSSIER}/bands.xlsx', f'{DOSSIER}/market.xlsx')
for etape in (former_cohortes, apply_rulebook, ml_anomaly, apply_ml_strong_signal, aggregate_risk, recommendations):
    detection = etape(detection, regles)

# Le résultat complet de la détection, enregistré à côté du notebook
detection.round(2).to_excel('anomalies.xlsx', index=False)
print(f"{len(detection)} salariés analysés. Résultat de la détection enregistré dans anomalies.xlsx")

# Combien de salariés dans chaque priorité, de la plus grave à la moins grave
PRIORITES = list(regles['prioritization_buckets'])[::-1]
display(detection['Severity'].value_counts().reindex(PRIORITES).fillna(0).astype(int).to_frame('Salariés'))

---
## 2. Choisir les anomalies à retirer

`A_RETIRER` est la liste des priorités qu'on retire. Pour en retirer plus, ajouter une priorité dans la liste, puis relancer à partir d'ici.

Avec une liste vide `[]`, personne n'est retiré.

In [ ]:
A_RETIRER = ['Critical']   # pour en retirer plus : ['Critical', 'Major'], ou ['Critical', 'Major', 'Minor']

inconnues = [p for p in A_RETIRER if p not in PRIORITES]
if inconnues:
    print(f"Priorité inconnue, ignorée : {inconnues}. Les priorités possibles : {PRIORITES}")

retires = detection[detection['Severity'].isin(A_RETIRER)]
print(f"{len(retires)} salariés à retirer sur {len(detection)} ({len(retires) / len(detection):.1%}).")
if retires.empty:
    print("Aucun salarié n'a cette priorité : personne n'est retiré, et la suite se fait sur tout le monde.")

# La liste des salariés retirés, avec tout ce que la détection dit d'eux, enregistrée à côté du notebook
retires.round(2).to_excel('salaries_retires.xlsx', index=False)
print("Liste enregistrée dans salaries_retires.xlsx")
a_montrer = ['Matricule', 'Job_Family', 'Niveau', 'Fixe_Annuel_MAD', 'Min', 'Mid', 'RiskScore', 'Severity', 'Reason_Principale']
display(retires[[c for c in a_montrer if c in retires.columns]].sort_values('RiskScore', ascending=False).head(10))

---
## 3. Le tableau de la régression

Le tableau sorti de la détection sert directement à la régression : chaque salarié y a déjà sa grille, son marché et ses ratios. On lui ajoute seulement ce que `Prediction_Compa_Ratio` calcule en plus, et on repère les salariés à retirer.

In [ ]:
CIBLES = ['Fixe_Annuel_MAD', 'Compa_Ratio']   # les deux variables à prédire : le salaire fixe et le Compa-Ratio

# 1. On part du tableau de la détection. Il a déjà Min, Mid, Max, la médiane du marché (Market_Median),
#    le Compa-Ratio (sous le nom CompaRatio) et Ratio_Grille_Marche (Mid / médiane du marché).
df = detection.copy()

# 2. Ce que Prediction_Compa_Ratio calcule en plus, avec les mêmes noms
df['Compa_Ratio'] = df['CompaRatio']
df['Amplitude_Grille'] = (df['Max'] - df['Min']) / df['Mid']
if 'Age' in df.columns and 'Anciennete' in df.columns:
    df['Age_Embauche'] = df['Age'] - df['Anciennete']
# Une division par zéro donne « infini » : on remet la case à vide
df = df.replace([np.inf, -np.inf], np.nan)
# Un ratio négatif ou nul est impossible (salaire à 0 ou négatif, grille dont le Min dépasse le Max) : case vidée aussi
for c in ['Compa_Ratio', 'Ratio_Grille_Marche', 'Amplitude_Grille']:
    df.loc[df[c] <= 0, c] = np.nan

# 3. Un salarié sans Compa-Ratio (pas de grille, ou pas de salaire) ne peut pas servir : on le retire
sans_cible = df['Compa_Ratio'].isna()
df = df[~sans_cible].reset_index(drop=True)
print(f"{len(df)} salariés gardés, {int(sans_cible.sum())} retirés (pas de grille ou pas de salaire).")

# 4. Une colonne vide, ou qui a la même valeur pour tout le monde, ne peut rien expliquer : on la retire.
#    Les nombres à virgule sont arrondis au millième : un écart d'arrondi (0,6600001) n'est pas une vraie différence.
inutiles = [c for c in df.columns if (df[c].round(3) if pd.api.types.is_float_dtype(df[c]) else df[c]).nunique() <= 1]
df = df.drop(columns=inutiles)
if inutiles:
    print(f"Colonnes vides ou identiques pour tous, retirées : {inutiles}")

# 5. Les salariés à retirer, repérés par leur matricule. True : le salarié fait partie des anomalies choisies.
anomalie = df['Matricule'].isin(retires['Matricule'])
print(f"Dans ce tableau : {int(anomalie.sum())} salariés à retirer, {int((~anomalie).sum())} qui restent.")

---
## 4. Les colonnes du modèle

La même liste que dans `Prediction_Compa_Ratio`. Un `#` devant une colonne la met de côté.

In [ ]:
COLONNES = [
    'Niveau',                  # le niveau du poste
    'Niveau_Etude_Max',        # le niveau d'études
    #'Cat_Ecole_Max',           # la catégorie d'école
    'Age_Embauche',            # âge − ancienneté
    'Anciennete',              # l'ancienneté, en années
    'Competence_N1',           # la note de compétence de l'an dernier
    'Performance_N1',          # la performance de l'an dernier
    'Hot_job',                 # la tension du métier sur le marché
    'Ratio_Grille_Marche',     # Mid / médiane du marché
    'Amplitude_Grille',        # (Max − Min) / Mid
]

# 1. On part du tableau complet (df) et on ne garde que les deux cibles et les colonnes de la liste
absentes = [c for c in COLONNES if c not in df.columns]
if absentes:
    print(f"Colonnes de la liste ignorées (absentes du fichier, ou identiques pour tous les salariés) : {absentes}")
donnees = df[CIBLES + [c for c in COLONNES if c in df.columns and c not in CIBLES]].copy()

# 2. La performance et le niveau d'études passent en rang.
# Un libellé inconnu compte comme une case vide : il prend le rang médian.
# Si la plupart des libellés du fichier ne sont pas ceux prévus, on ne force rien : la colonne reste
# en texte et n'est pas utilisée.
RANG_PERFORMANCE = {"Très en dessous": 1, "Proche de l'objectif": 2, "A l'objectif": 3, "Supérieur à l'objectif": 4, "Très au-delà": 5}
RANG_ETUDE = {'SECONDAIRE': 0, 'BAC': 1, 'BAC+1': 2, 'BAC+2': 3, 'BAC+3': 4, 'BAC+4': 5, 'BAC+5': 6, 'BAC+6': 6,
              'BAC+7': 7, 'EXPERTISE COMPTABLE': 7}
for colonne, rangs in [('Performance_N1', RANG_PERFORMANCE), ('Niveau_Etude_Max', RANG_ETUDE)]:
    if colonne in donnees.columns and donnees[colonne].isin(list(rangs)).mean() > 0.5:
        donnees[colonne] = donnees[colonne].map(rangs)
        donnees[colonne] = donnees[colonne].fillna(donnees[colonne].median())

# 3. Les colonnes restées en texte ne sont pas utilisées : les modèles ne lisent que des chiffres
en_texte = list(donnees.select_dtypes(exclude='number').columns)
if en_texte:
    print(f"Colonnes en texte non utilisées : {en_texte}")
    donnees = donnees.drop(columns=en_texte)

# 4. Un salarié qui a une case vide dans une des colonnes choisies sort : le modèle ne sait pas lire une case vide
vides = donnees.isna().any(axis=1)
donnees = donnees[~vides]
print(f"{len(donnees)} salariés pour la suite ({int(vides.sum())} retirés pour une case vide dans une des colonnes choisies).")

# 5. Les colonnes que les modèles reçoivent : celles de la liste, sans les deux cibles
X = donnees.drop(columns=CIBLES)
print(f"{X.shape[1]} colonnes : {list(X.columns)}")

---
## 5. Les six modèles, sans les anomalies

Les mêmes six modèles que dans `Prediction_Compa_Ratio`, avec les mêmes réglages : M1, M2 et M5 prédisent le salaire, M3, M4 et M6 le Compa-Ratio ; une régression linéaire, un arbre de décision, un gradient boosting.

On coupe en train (80 %) et test (20 %), puis les anomalies choisies sortent des deux.

Chaque modèle apprend deux fois :

* **avec les anomalies** : sur tout le train, anomalies comprises ;
* **sans les anomalies** : sur le train dont les anomalies sont sorties. C'est le résultat demandé.

Les deux sont jugés sur les mêmes salariés : ceux du test, sans les anomalies. La seule chose qui change entre les deux colonnes est donc ce que le modèle a vu pour apprendre.

Pourquoi les mêmes salariés : une anomalie est par définition un salaire que le modèle prédit mal. Si on la laissait dans le test d'un côté et pas de l'autre, l'erreur baisserait toute seule, sans que le modèle soit meilleur.

In [ ]:
# 1. On coupe en train et test, une seule fois. random_state : le même découpage à chaque lancement.
train, test = train_test_split(X.index, test_size=0.2, random_state=42)
# Les anomalies choisies sortent du train et du test
train_sans, test_sans = train[~anomalie[train].to_numpy()], test[~anomalie[test].to_numpy()]
print(f"Train : {len(train)} salariés, {len(train_sans)} sans les anomalies.")
print(f"Test  : {len(test)} salariés, {len(test_sans)} sans les anomalies.")
if len(train_sans) == len(train):
    print("Aucune anomalie dans le train : les deux apprentissages sont les mêmes, les deux colonnes du tableau aussi.")

# 2. Les six modèles et leurs réglages, comme dans Prediction_Compa_Ratio.
#    PROFONDEUR : le nombre de questions que l'arbre peut enchaîner.
PROFONDEUR = 5
# LOG_SALAIRE. True : M1, M2 et M5 apprennent le logarithme du salaire. False : le salaire tel quel.
LOG_SALAIRE = True
# LOG_COMPA_RATIO. True : M3, M4 et M6 apprennent le logarithme du Compa-Ratio. False : le Compa-Ratio tel quel.
LOG_COMPA_RATIO = True
salaire = 'le salaire (en log)' if LOG_SALAIRE else 'le salaire'
compa_ratio = 'le Compa-Ratio (en log)' if LOG_COMPA_RATIO else 'le Compa-Ratio'
MODELES = {
    'M1': (f'{salaire}, par une régression', 'Fixe_Annuel_MAD', LinearRegression()),
    'M2': (f'{salaire}, par un arbre', 'Fixe_Annuel_MAD', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    'M3': (f'{compa_ratio}, par une régression', 'Compa_Ratio', LinearRegression()),
    'M4': (f'{compa_ratio}, par un arbre', 'Compa_Ratio', DecisionTreeRegressor(max_depth=PROFONDEUR, random_state=42)),
    'M5': (f'{salaire}, par un gradient boosting', 'Fixe_Annuel_MAD', HistGradientBoostingRegressor(random_state=42)),
    'M6': (f'{compa_ratio}, par un gradient boosting', 'Compa_Ratio', HistGradientBoostingRegressor(random_state=42)),
}
# Les modèles du salaire reçoivent en plus le Mid : sans lui, ils ne connaissent pas la grille du poste.
# N'y mettre que des montants : avec LOG_SALAIRE = True, ces colonnes passent elles aussi en logarithme.
EN_PLUS_POUR_LE_SALAIRE = ['Mid']


def salaire_predit(modele, cible, apprend, verifie):
    # Le modèle apprend sur les salariés « apprend », puis prédit le salaire (en MAD) des salariés « verifie ».
    # clone : un modèle neuf à chaque fois, qui n'a rien gardé de l'apprentissage d'avant.
    modele = clone(modele)
    if cible == 'Fixe_Annuel_MAD':
        en_plus = [c for c in EN_PLUS_POUR_LE_SALAIRE if c in df.columns and c not in X.columns]
        if LOG_SALAIRE:
            X_modele = X.join(np.log(df[en_plus]).add_prefix('Log_'))
            modele.fit(X_modele.loc[apprend], np.log(df.loc[apprend, cible]))
            return np.exp(modele.predict(X_modele.loc[verifie]))
        X_modele = X.join(df[en_plus])
        modele.fit(X_modele.loc[apprend], df.loc[apprend, cible])
        return modele.predict(X_modele.loc[verifie])
    # Les modèles du Compa-Ratio : salaire prédit = Compa-Ratio prédit × Mid
    if LOG_COMPA_RATIO:
        modele.fit(X.loc[apprend], np.log(df.loc[apprend, cible]))
        return np.exp(modele.predict(X.loc[verifie])) * df.loc[verifie, 'Mid']
    modele.fit(X.loc[apprend], df.loc[apprend, cible])
    return modele.predict(X.loc[verifie]) * df.loc[verifie, 'Mid']


def metriques(cible, y_pred):
    # MAE et RMSE sur le salaire, en MAD. R² sur ce que le modèle prédit : le salaire, ou le Compa-Ratio.
    y_test, mid = df.loc[test_sans, 'Fixe_Annuel_MAD'], df.loc[test_sans, 'Mid']
    return {'MAE (MAD)': mean_absolute_error(y_test, y_pred), 'RMSE (MAD)': mean_squared_error(y_test, y_pred) ** 0.5,
            'R²': r2_score(y_test, y_pred) if cible == 'Fixe_Annuel_MAD' else r2_score(y_test / mid, y_pred / mid)}


# 3. Chaque modèle apprend avec les anomalies, puis sans. Les deux prédisent les mêmes salariés : le test sans les anomalies.
#    Les prédictions gardées (tableau « resultats ») sont celles du modèle appris sans les anomalies.
resultats = pd.DataFrame({'y_test_MAD': df.loc[test_sans, 'Fixe_Annuel_MAD']})
avec, sans = {}, {}
for nom, (quoi, cible, modele) in MODELES.items():
    avec[f'{nom} : {quoi}'] = metriques(cible, salaire_predit(modele, cible, train, test_sans))
    y_pred = salaire_predit(modele, cible, train_sans, test_sans)
    sans[f'{nom} : {quoi}'] = metriques(cible, y_pred)
    resultats[f'{nom}_y_pred_MAD'] = y_pred
    resultats[f'{nom}_erreur_MAD'] = resultats['y_test_MAD'] - resultats[f'{nom}_y_pred_MAD']

# 4. Les métriques des six modèles, côte à côte
comparaison = pd.concat({'appris avec les anomalies': pd.DataFrame(avec).T, 'appris sans les anomalies': pd.DataFrame(sans).T}, axis=1)
comparaison = comparaison.swaplevel(axis=1)[[(m, a) for m in ['MAE (MAD)', 'RMSE (MAD)', 'R²'] for a in comparaison.columns.levels[0]]]
display(comparaison.round(2))

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : sur ce que le modèle prédit. Le salaire pour M1, M2 et M5 ; le Compa-Ratio pour M3, M4 et M6. Les deux ne se comparent pas entre eux.

Lecture : si la MAE « sans les anomalies » est plus basse que la MAE « avec », les anomalies gênaient le modèle. Si les deux sont presque égales, les retirer ne change rien à ce que le modèle apprend.

La détection ne retient que les salaires trop bas. En les retirant, le modèle apprend sur des salaires un peu plus hauts : ses prédictions montent légèrement.

In [ ]:
# Les graphes sont ceux des modèles appris sans les anomalies.
# Le salaire prédit face au vrai, pour les six modèles : un bon modèle met les points le long de la diagonale
points = pd.concat([pd.DataFrame({'Modèle': f'{nom} : {quoi}', 'y_test': resultats['y_test_MAD'],
                                  'y_pred': resultats[f'{nom}_y_pred_MAD'], 'erreur': resultats[f'{nom}_erreur_MAD']})
                    for nom, (quoi, cible, modele) in MODELES.items()])
fig = px.scatter(points, x='y_test', y='y_pred', facet_col='Modèle', facet_col_wrap=2, opacity=0.4, height=1300,
                 title='Salaire fixe prédit face au vrai (test, sans les anomalies)',
                 labels={'y_test': 'Vrai salaire fixe (y_test)', 'y_pred': 'Salaire fixe prédit (y_pred)'})
bornes = [points['y_test'].min(), points['y_test'].max()]
fig.add_scatter(x=bornes, y=bornes, mode='lines', line=dict(color='black'), showlegend=False, row='all', col='all')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

# Les erreurs sur le salaire : un bon modèle les serre autour de 0
fig = px.histogram(points, x='erreur', facet_col='Modèle', facet_col_wrap=2, nbins=60, height=1000,
                   title='Répartition des erreurs sur le salaire fixe (test, sans les anomalies)', labels={'erreur': 'Erreur (y_test − y_pred)'})
fig.add_vline(x=0, line_color='black')
fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Modèle=', '')))
fig.show()

In [ ]:
# Export en Excel : les salariés du test, sans les anomalies (toutes les colonnes du tableau, matricule compris),
# avec à côté le vrai salaire (y_test), puis le salaire prédit (y_pred) et l'erreur de chacun des six modèles.
export = df.loc[resultats.index].join(resultats).round(2)   # 2 chiffres après la virgule dans le fichier
# Le fichier s'écrit à côté du notebook
export.to_excel('resultats_prediction.xlsx', index=False)
print(f"{len(export)} salariés du test et {export.shape[1]} colonnes exportés dans resultats_prediction.xlsx")
export.head()